# Demand Forecasting and Inventory Optimization Agent

## 01 - Data Understanding

### Objective

The purpose of this notebook is to understand the structure, quality,
and characteristics of the given retail dataset.

At this stage, we are NOT building a forecasting model.

We will investigate the dataset to understand:

1. What is happening?
2. Where is it happening?
3. When is it happening?
4. What patterns do we observe?
5. What variables are associated with those patterns?
6. What inventory problem can be identified from the data?

### Approach

We will use the given dataset as the primary source of evidence.
No assumptions will be made before examining the data.

In [1]:
import pandas as pd
df = pd.read_csv("../data/sales_data.csv")
print("Dataset loaded successfully!")

Dataset loaded successfully!


In [3]:
print("Number of rows:", df.shape[0])
print("Number of columns:", df.shape[1])

Number of rows: 76000
Number of columns: 16


In [45]:
print(df.columns)

Index(['Date', 'Store ID', 'Product ID', 'Category', 'Region',
       'Inventory Level', 'Units Sold', 'Units Ordered', 'Price', 'Discount',
       'Weather Condition', 'Promotion', 'Competitor Pricing', 'Seasonality',
       'Epidemic', 'Demand', 'Demand_Sales_Difference',
       'Demand_Inventory_Difference', 'Year', 'Month', 'DayOfWeek'],
      dtype='object')


In [6]:
df.dtypes

Date                   object
Store ID               object
Product ID             object
Category               object
Region                 object
Inventory Level         int64
Units Sold              int64
Units Ordered           int64
Price                 float64
Discount                int64
Weather Condition      object
Promotion               int64
Competitor Pricing    float64
Seasonality            object
Epidemic                int64
Demand                  int64
dtype: object

In [7]:
missing_values = df.isnull().sum()

print(missing_values)

Date                  0
Store ID              0
Product ID            0
Category              0
Region                0
Inventory Level       0
Units Sold            0
Units Ordered         0
Price                 0
Discount              0
Weather Condition     0
Promotion             0
Competitor Pricing    0
Seasonality           0
Epidemic              0
Demand                0
dtype: int64


In [8]:
duplicate_count = df.duplicated().sum()

print("Duplicate rows:", duplicate_count)

Duplicate rows: 0


In [4]:
df.describe()

,Inventory Level,Units Sold,Units Ordered,Price,Discount,Promotion,Competitor Pricing,Epidemic,Demand
count,76000.000000,76000.000000,76000.000000,76000.000000,76000.000000,76000.000000,76000.000000,76000.000000,76000.000000
mean,301.062842,88.827316,89.090645,67.726028,9.087039,0.328947,69.454029,0.200000,104.317158
std,226.510161,43.994525,162.404627,39.377899,7.475781,0.469834,40.943818,0.400003,46.964801
min,0.000000,0.000000,0.000000,4.740000,0.000000,0.000000,4.290000,0.000000,4.000000
25%,136.000000,58.000000,0.000000,31.997500,5.000000,0.000000,32.620000,0.000000,71.000000
50%,227.000000,84.000000,0.000000,64.500000,10.000000,0.000000,65.700000,0.000000,100.000000
75%,408.000000,114.000000,121.000000,95.830000,10.000000,1.000000,97.932500,0.000000,133.000000
max,2267.000000,426.000000,1616.000000,228.030000,25.000000,1.000000,261.220000,1.000000,430.000000


Finding the relation between demand and the units sold

In [9]:
df["Demand_Sales_Difference"] = df["Demand"] - df["Units Sold"]

df[["Demand", "Units Sold", "Demand_Sales_Difference"]].head(10)

,Demand,Units Sold,Demand_Sales_Difference
0,115,102,13
1,229,117,112
2,157,114,43
3,52,45,7
4,59,65,-6
5,55,60,-5
6,94,81,13
7,61,42,19
8,129,88,41
9,69,70,-1


In [10]:
print("Demand > Units Sold:",
      (df["Demand"] > df["Units Sold"]).sum())

print("Demand = Units Sold:",
      (df["Demand"] == df["Units Sold"]).sum())

print("Demand < Units Sold:",
      (df["Demand"] < df["Units Sold"]).sum())

Demand > Units Sold: 53440
Demand = Units Sold: 1560
Demand < Units Sold: 21000


In [11]:
total = len(df)

greater = (df["Demand"] > df["Units Sold"]).sum()
equal = (df["Demand"] == df["Units Sold"]).sum()
less = (df["Demand"] < df["Units Sold"]).sum()

print(f"Demand > Units Sold: {greater} ({greater/total*100:.2f}%)")
print(f"Demand = Units Sold: {equal} ({equal/total*100:.2f}%)")
print(f"Demand < Units Sold: {less} ({less/total*100:.2f}%)")

Demand > Units Sold: 53440 (70.32%)
Demand = Units Sold: 1560 (2.05%)
Demand < Units Sold: 21000 (27.63%)


In [12]:
df["Demand_Sales_Difference"].describe()

count    76000.000000
mean        15.489842
std         26.404353
min        -68.000000
25%         -2.000000
50%         11.000000
75%         27.000000
max        316.000000
Name: Demand_Sales_Difference, dtype: float64

## Investigating Inventory When Demand Exceeds Units Sold

In [13]:
mismatch = df[df["Demand"] > df["Units Sold"]]

mismatch["Inventory Level"].describe()

count    53440.000000
mean       287.706044
std        225.157703
min          0.000000
25%        126.000000
50%        211.000000
75%        391.000000
max       2267.000000
Name: Inventory Level, dtype: float64

In [14]:
df["Demand_Inventory_Difference"] = (
    df["Demand"] - df["Inventory Level"]
)

df[[
    "Demand",
    "Inventory Level",
    "Demand_Inventory_Difference"
]].head(20)

,Demand,Inventory Level,Demand_Inventory_Difference
0,115,195,-80
1,229,117,112
2,157,247,-90
3,52,139,-87
4,59,152,-93
5,55,209,-154
6,94,118,-24
7,61,244,-183
8,129,115,14
9,69,192,-123


In [15]:
demand_greater_inventory = (
    df["Demand"] > df["Inventory Level"]
).sum()

demand_equal_inventory = (
    df["Demand"] == df["Inventory Level"]
).sum()

demand_less_inventory = (
    df["Demand"] < df["Inventory Level"]
).sum()

print("Demand > Inventory:", demand_greater_inventory)
print("Demand = Inventory:", demand_equal_inventory)
print("Demand < Inventory:", demand_less_inventory)

Demand > Inventory: 10394
Demand = Inventory: 201
Demand < Inventory: 65405


In [16]:
total = len(df)

print(
    f"Demand > Inventory: "
    f"{demand_greater_inventory} "
    f"({demand_greater_inventory / total * 100:.2f}%)"
)

print(
    f"Demand = Inventory: "
    f"{demand_equal_inventory} "
    f"({demand_equal_inventory / total * 100:.2f}%)"
)

print(
    f"Demand < Inventory: "
    f"{demand_less_inventory} "
    f"({demand_less_inventory / total * 100:.2f}%)"
)

Demand > Inventory: 10394 (13.68%)
Demand = Inventory: 201 (0.26%)
Demand < Inventory: 65405 (86.06%)


In [17]:
shortage_like = df[
    df["Demand"] > df["Inventory Level"]
]

shortage_like["Demand_Inventory_Difference"].describe()

count    10394.000000
mean        45.790648
std         39.269678
min          1.000000
25%         16.000000
50%         35.000000
75%         66.000000
max        316.000000
Name: Demand_Inventory_Difference, dtype: float64

In [18]:
shortage_like[
    [
        "Demand",
        "Inventory Level",
        "Units Sold",
        "Units Ordered"
    ]
].head(20)

,Demand,Inventory Level,Units Sold,Units Ordered
1,229,117,117,249
8,129,115,88,139
15,115,103,78,372
20,118,102,102,331
28,145,131,131,260
35,194,113,113,323
42,126,122,122,127
52,176,105,105,743
81,186,131,131,474
89,136,132,132,436


## Investigating the Size of the Demand-Inventory Gap

The previous analysis showed how often Demand is greater than,
equal to, or less than Inventory Level.

Now we want to understand the size of the gap when Demand
is greater than Inventory.

A small gap and a large gap may represent very different
inventory situations.


In [29]:
shortage_like = df[df["Demand"] > df["Inventory Level"]].copy()

shortage_like["Demand_Inventory_Difference"].describe()

count    10394.000000
mean        45.790648
std         39.269678
min          1.000000
25%         16.000000
50%         35.000000
75%         66.000000
max        316.000000
Name: Demand_Inventory_Difference, dtype: float64

## Investigating Demand, Inventory and Units Sold

We identified observations where Demand is greater than Inventory.

Now we investigate whether these observations are also associated
with Units Sold being lower than Demand.

This helps us determine whether the demand-inventory mismatch
is reflected in actual sales.

In [24]:
shortage_like[
    [
        "Demand",
        "Inventory Level",
        "Units Sold",
        "Units Ordered"
    ]
].describe()

,Demand,Inventory Level,Units Sold,Units Ordered
count,10394.000000,10394.000000,10394.000000,10394.000000
mean,141.320858,95.530210,90.889552,152.649990
std,45.529402,44.989132,42.379330,153.650997
min,6.000000,0.000000,0.000000,0.000000
25%,110.000000,68.000000,66.000000,71.000000
50%,137.000000,96.000000,91.000000,131.000000
75%,169.000000,123.000000,116.000000,185.000000
max,430.000000,339.000000,339.000000,1447.000000


In [25]:
demand_greater_than_sales = (
    shortage_like["Demand"] > shortage_like["Units Sold"]
).sum()

total_shortage_like = len(shortage_like)

print("Demand > Units Sold:", demand_greater_than_sales)
print(
    "Percentage:",
    f"{demand_greater_than_sales / total_shortage_like * 100:.2f}%"
)

Demand > Units Sold: 10394
Percentage: 100.00%


The above result tells us when 
Demand > Inventory, Demand is also > Units Sold

## Investigating the Demand-Sales Gap

We found that in every observation where Demand exceeds
Inventory Level, Units Sold is also lower than Demand.

Now we investigate the size of the Demand-Sales gap to
understand how large the difference is.

In [30]:
shortage_like["Demand_Sales_Difference"] = (
    shortage_like["Demand"] - shortage_like["Units Sold"]
)

shortage_like["Demand_Sales_Difference"].describe()

count    10394.000000
mean        50.431307
std         37.037250
min          1.000000
25%         24.000000
50%         42.000000
75%         67.750000
max        316.000000
Name: Demand_Sales_Difference, dtype: float64

Demand > Inventory
        ↓
10,394 observations
        ↓
Demand > Units Sold
        ↓
100% of those observations
        ↓
Average Demand–Sales gap ≈ 50.43 units

In [ ]:
## Stores which has the demand greater than inventory level
shortage_like["Store ID"].value_counts()

Store ID
S001    2293
S005    2145
S003    2016
S004    1975
S002    1965
Name: count, dtype: int64

Now we are fidning - For each store, what percentage of its observations have Demand > Inventory?

In [33]:
store_summary = df.groupby("Store ID").agg(
    Total_Observations=("Demand", "size"),
    Demand_Greater_Inventory=(
        "Demand_Inventory_Difference",
        lambda x: (x > 0).sum()
    )
)

store_summary["Percentage_Demand_Greater_Inventory"] = (
    store_summary["Demand_Greater_Inventory"]
    / store_summary["Total_Observations"]
    * 100
)

store_summary

,Total_Observations,Demand_Greater_Inventory,Percentage_Demand_Greater_Inventory
Store ID,,,
S001,15200,2293,15.085526
S002,15200,1965,12.927632
S003,15200,2016,13.263158
S004,15200,1975,12.993421
S005,15200,2145,14.111842


## Investigating Demand > Inventory and Products

In [34]:
product_summary = df.groupby("Product ID").agg(
    Total_Observations=("Demand", "size"),
    Demand_Greater_Inventory=(
        "Demand_Inventory_Difference",
        lambda x: (x > 0).sum()
    )
)

product_summary["Percentage_Demand_Greater_Inventory"] = (
    product_summary["Demand_Greater_Inventory"]
    / product_summary["Total_Observations"]
    * 100
)

product_summary.sort_values(
    "Percentage_Demand_Greater_Inventory",
    ascending=False
)

,Total_Observations,Demand_Greater_Inventory,Percentage_Demand_Greater_Inventory
Product ID,,,
P0008,3800,1079,28.394737
P0018,3800,869,22.868421
P0019,3800,773,20.342105
P0009,3800,706,18.578947
P0010,3800,691,18.184211
P0006,3800,654,17.210526
P0017,3800,647,17.026316
P0002,3800,538,14.157895
P0012,3800,465,12.236842


In [35]:
product_gap = shortage_like.groupby("Product ID").agg(
    Observations=("Demand", "size"),
    Average_Demand_Sales_Gap=("Demand_Sales_Difference", "mean"),
    Median_Demand_Sales_Gap=("Demand_Sales_Difference", "median")
)

product_gap.sort_values(
    "Average_Demand_Sales_Gap",
    ascending=False
)

,Observations,Average_Demand_Sales_Gap,Median_Demand_Sales_Gap
Product ID,,,
P0007,236,65.419492,52.5
P0013,386,62.733161,52.0
P0001,366,61.407104,52.0
P0011,272,58.875000,47.0
P0002,538,58.821561,48.5
P0016,417,57.798561,48.0
P0015,384,53.481771,43.0
P0009,706,53.470255,45.0
P0004,461,53.084599,43.0


Finding the products that have both a high frequency of Demand > Inventory AND a large Demand–Sales gap

In [36]:
product_analysis = product_summary[
    ["Total_Observations", "Demand_Greater_Inventory",
     "Percentage_Demand_Greater_Inventory"]
].join(
    product_gap[
        ["Average_Demand_Sales_Gap", "Median_Demand_Sales_Gap"]
    ]
)

product_analysis.sort_values(
    "Percentage_Demand_Greater_Inventory",
    ascending=False
)

,Total_Observations,Demand_Greater_Inventory,Percentage_Demand_Greater_Inventory,Average_Demand_Sales_Gap,Median_Demand_Sales_Gap
Product ID,,,,,
P0008,3800,1079,28.394737,44.740500,38.0
P0018,3800,869,22.868421,46.331415,42.0
P0019,3800,773,20.342105,42.304010,36.0
P0009,3800,706,18.578947,53.470255,45.0
P0010,3800,691,18.184211,50.109986,43.0
P0006,3800,654,17.210526,48.906728,41.5
P0017,3800,647,17.026316,43.896445,39.0
P0002,3800,538,14.157895,58.821561,48.5
P0012,3800,465,12.236842,46.008602,38.0


## When is this pattern happening?

In [37]:
df["Date"].min(), df["Date"].max()

('2022-01-01', '2024-01-30')

In [38]:
df["Date"] = pd.to_datetime(df["Date"])

df["Year"] = df["Date"].dt.year
df["Month"] = df["Date"].dt.month
df["DayOfWeek"] = df["Date"].dt.day_name()

df[["Date", "Year", "Month", "DayOfWeek"]].head()

,Date,Year,Month,DayOfWeek
0,2022-01-01,2022,1,Saturday
1,2022-01-01,2022,1,Saturday
2,2022-01-01,2022,1,Saturday
3,2022-01-01,2022,1,Saturday
4,2022-01-01,2022,1,Saturday


In [39]:
monthly_mismatch = df.groupby("Month").agg(
    Total_Observations=("Demand", "size"),
    Demand_Greater_Inventory=(
        "Demand_Inventory_Difference",
        lambda x: (x > 0).sum()
    )
)

monthly_mismatch["Percentage"] = (
    monthly_mismatch["Demand_Greater_Inventory"]
    / monthly_mismatch["Total_Observations"]
    * 100
)

monthly_mismatch

,Total_Observations,Demand_Greater_Inventory,Percentage
Month,,,
1,9200,1339,14.554348
2,5600,651,11.625000
3,6200,922,14.870968
4,6000,720,12.000000
5,6200,713,11.500000
6,6000,933,15.550000
7,6200,808,13.032258
8,6200,927,14.951613
9,6000,830,13.833333


In [40]:
yearly_mismatch = df.groupby("Year").agg(
    Total_Observations=("Demand", "size"),
    Demand_Greater_Inventory=(
        "Demand_Inventory_Difference",
        lambda x: (x > 0).sum()
    )
)

yearly_mismatch["Percentage"] = (
    yearly_mismatch["Demand_Greater_Inventory"]
    / yearly_mismatch["Total_Observations"]
    * 100
)

yearly_mismatch

,Total_Observations,Demand_Greater_Inventory,Percentage
Year,,,
2022,36500,5060,13.863014
2023,36500,4936,13.523288
2024,3000,398,13.266667


In [41]:
monthly_demand = df.groupby("Month").agg(
    Average_Demand=("Demand", "mean"),
    Median_Demand=("Demand", "median"),
    Minimum_Demand=("Demand", "min"),
    Maximum_Demand=("Demand", "max")
)

monthly_demand

,Average_Demand,Median_Demand,Minimum_Demand,Maximum_Demand
Month,,,,
1,106.040000,103.0,4,339
2,92.882500,89.0,4,315
3,113.613871,109.0,4,361
4,92.816667,89.0,4,283
5,86.521129,82.0,4,302
6,117.346000,111.0,4,430
7,101.561613,96.0,4,337
8,119.803387,113.0,4,378
9,107.431500,103.0,4,327


Finding if the demand pattern repeat across different years?

In [43]:
year_month_demand = df.groupby(
    ["Year", "Month"]
).agg(
    Average_Demand=("Demand", "mean")
).reset_index()

year_month_demand

,Year,Month,Average_Demand
0,2022,1,110.175484
1,2022,2,114.243929
2,2022,3,113.926774
3,2022,4,101.692000
4,2022,5,77.132903
5,2022,6,115.241667
6,2022,7,83.785806
7,2022,8,119.495806
8,2022,9,108.088000
9,2022,10,106.408065


In [44]:
day_demand = (
    df.groupby("DayOfWeek")["Demand"]
    .agg(
        Average_Demand="mean",
        Median_Demand="median",
        Minimum_Demand="min",
        Maximum_Demand="max"
    )
    .sort_values("Average_Demand", ascending=False)
)

day_demand

,Average_Demand,Median_Demand,Minimum_Demand,Maximum_Demand
DayOfWeek,,,,
Wednesday,105.192037,101.0,4,430
Sunday,104.662294,101.0,4,373
Saturday,104.484037,100.0,4,370
Thursday,104.408333,100.0,4,369
Friday,104.394444,101.0,4,343
Tuesday,103.665229,99.0,4,337
Monday,103.423303,99.0,4,378


## Price → Demand

In [46]:
price_demand = (
    df.groupby("Price")["Demand"]
    .agg(
        Average_Demand="mean",
        Median_Demand="median",
        Observations="count"
    )
    .sort_index()
)

price_demand.head(20)

,Average_Demand,Median_Demand,Observations
Price,,,
4.74,28.0,28.0,1
4.83,33.0,33.0,1
5.04,31.0,31.0,1
5.25,57.0,57.0,1
5.27,41.0,41.0,1
5.31,74.0,74.0,1
5.32,74.0,74.0,1
5.38,30.0,30.0,1
5.42,46.0,46.0,1


In [47]:
price_demand_correlation = df["Price"].corr(df["Demand"])

print("Price-Demand Correlation:", price_demand_correlation)

Price-Demand Correlation: -0.023460821957205256


## Discount → Demand

In [48]:
discount_demand = (
    df.groupby("Discount")["Demand"]
    .agg(
        Average_Demand="mean",
        Median_Demand="median",
        Observations="count"
    )
    .sort_index()
)

discount_demand

,Average_Demand,Median_Demand,Observations
Discount,,,
0,94.748861,92.0,17126
5,94.941896,92.0,16918
10,102.767834,99.0,23298
15,123.575981,120.0,6245
20,124.065256,120.0,6191
25,122.967374,120.0,6222


In [49]:
discount_demand_correlation = df["Discount"].corr(df["Demand"])

print("Discount-Demand Correlation:", discount_demand_correlation)

Discount-Demand Correlation: 0.22472264702914563


# Summary of Data Understanding

## Key Findings

1. The dataset contains historical retail observations from January 2022 to January 2024.

2. Demand is frequently greater than the available Inventory Level, indicating potential inventory shortage situations.

3. When Demand exceeds Inventory Level, Units Sold is also lower than Demand, indicating that inventory constraints may be associated with unrealized sales.

4. The Demand-Sales gap is therefore an important indicator for identifying potential lost-sales situations.

5. The frequency of Demand > Inventory varies across products and stores. Some products show a substantially higher mismatch frequency than others.

6. Demand also varies across months and years, indicating the presence of temporal patterns that should be considered during forecasting.

7. Demand varies somewhat across days of the week, although the differences are relatively small compared with the variation observed across months and products.

8. Price has a very weak linear relationship with Demand, with a correlation of approximately -0.023.

9. Discount has a positive but relatively weak relationship with Demand, with a correlation of approximately 0.225.

## Problem Identified

The analysis indicates a potential demand-supply mismatch:

Demand > Inventory
        ↓
Inventory may be insufficient
        ↓
Units Sold < Demand
        ↓
Potential lost sales

The next stage will investigate the dataset more deeply to identify the major factors associated with demand and inventory mismatches.